In [3]:
import os
import operator
from typing import TypedDict, Annotated, Optional, get_type_hints
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
llm = ChatOpenAI(model="gpt-4o-mini")

In [4]:
# reducer
def my_reducer(cur, new):
    return (cur or []) + new

class AnnotatedState(TypedDict):
    items : Annotated[list, my_reducer]
    
hints = get_type_hints(AnnotatedState, include_extras= True)
hints

{'items': typing.Annotated[list, <function my_reducer at 0x10bcbaef0>]}

In [5]:
import operator
class AccumState(TypedDict):
    msgs : Annotated[list, operator.add]

def node_a(state):
    return {'msgs' : ['A 추가']}
def node_b(state):
    return {'msgs' : ['B 추가']}

b = StateGraph(AccumState)
b.add_node('node_a', node_a)
b.add_node('node_b', node_b)

b.add_edge(START, 'node_a')
b.add_edge('node_a', 'node_b')
b.add_edge('node_b', END)

app = b.compile()
app.invoke({'msgs': []})

{'msgs': ['A 추가', 'B 추가']}

In [7]:
class ScoreState(TypedDict):
    scores: Annotated[list, operator.add]

def add_kor(state):
    return {"scores": [80]}

def add_eng(state):
    return {"scores": [90]}

def add_math(state):
    return {"scores": [70]}

b = StateGraph(ScoreState)

b.add_node("add_kor", add_kor)
b.add_node("add_eng", add_eng)
b.add_node("add_math", add_math)

b.add_edge(START, "add_kor")
b.add_edge("add_kor", "add_eng")
b.add_edge("add_eng", "add_math")
b.add_edge("add_math", END)

app = b.compile()
result = app.invoke({"scores": []})
print(result)

{'scores': [80, 90, 70]}


In [8]:
from langgraph.graph.message import add_messages

In [9]:
class ChatState(TypedDict):
    messages : Annotated[list, add_messages]

def user_says(state):
    return {'messages' : [HumanMessage(content='안녕')]}

def bot_says(state):
    return {'messages' : [AIMessage(content='반갑습니다')]}

b = StateGraph(ChatState)
b.add_node('user_says', user_says)
b.add_node('bot_says', bot_says)

b.add_edge(START, 'user_says')
b.add_edge('user_says', 'bot_says')
b.add_edge('bot_says', END)

app = b.compile()
result = app.invoke({"messages": []})
print(result)

{'messages': [HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}, id='a92d3d20-23b8-4b0f-99d8-06626886c364'), AIMessage(content='반갑습니다', additional_kwargs={}, response_metadata={}, id='2168d7ce-2654-4951-b0d2-3ef95c651d90', tool_calls=[], invalid_tool_calls=[])]}


In [10]:
sys_m = SystemMessage(content='당신은 컴퓨터 전문가입니다.')
hum_m = HumanMessage(content='랭그래프?')
ai_m = AIMessage(content='그래프 기반 라이브러리')
tool_m = ToolMessage(content='검색결과...', tool_call_id='call_abc123')

In [11]:
for m in [sys_m, hum_m, ai_m, tool_m]:
    print(f"type={m.type} | class={type(m).__name__}")

type=system | class=SystemMessage
type=human | class=HumanMessage
type=ai | class=AIMessage
type=tool | class=ToolMessage


In [12]:
add_messages(sys_m, hum_m)

[SystemMessage(content='당신은 컴퓨터 전문가입니다.', additional_kwargs={}, response_metadata={}, id='24939c0b-2638-4f2e-b76d-5774ff0759c5'),
 HumanMessage(content='랭그래프?', additional_kwargs={}, response_metadata={}, id='d6ae0c5b-3de3-40a2-a52e-857e60e7edb9')]

In [13]:
ai_with_tools = AIMessage(
    content="",
    tool_calls=[
        {'name' : 'search', 'args' : {'q': '랭그래프'}, 'id' : 'call_001'},
        {'name' : 'calc', 'args' : {'x': 7, 'y':3}, 'id' : 'call_002'}
    ]
)

In [14]:
ai_with_tools

AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'search', 'args': {'q': '랭그래프'}, 'id': 'call_001', 'type': 'tool_call'}, {'name': 'calc', 'args': {'x': 7, 'y': 3}, 'id': 'call_002', 'type': 'tool_call'}], invalid_tool_calls=[])

In [15]:
def merge_dicts(cur, new):
    return {**(cur or {}), **(new or {})}

class MetaState(TypedDict):
    meta : Annotated[dict, merge_dicts]

def fetch_user(state): return {'meta' : {'user_id' : 'u123'}}
def fetch_session(state): return {'meta' : {'session_id' : 's456'}}
def fetch_locale(state): return {'meta' : {'locale' : 'ko-KR'}}

b = StateGraph(MetaState)
b.add_node('fetch_user', fetch_user)
b.add_node('fetch_session', fetch_session)
b.add_node('fetch_locale', fetch_locale)

b.add_edge(START, 'fetch_user')
b.add_edge('fetch_user', 'fetch_session')
b.add_edge('fetch_session', 'fetch_locale')
b.add_edge('fetch_locale', END)

app = b.compile()
result = app.invoke({"meta": []})
print(result)

{'meta': {'user_id': 'u123', 'session_id': 's456', 'locale': 'ko-KR'}}


In [17]:
class ConfigState(TypedDict):
    config: Annotated[dict, merge_dicts]


def set_model(state):
    return {"config": {"model": "gpt-4o-mini"}}


def set_temperature(state):
    return {"config": {"temperature": 0.5}}


def set_override_model(state):
    return {"config": {"model": "gpt-4o"}}


builder = StateGraph(ConfigState)
builder.add_node("set_model", set_model)
builder.add_node("set_temperature", set_temperature)
builder.add_node("set_override_model", set_override_model)

builder.add_edge(START, "set_model")
builder.add_edge("set_model", "set_temperature")
builder.add_edge("set_temperature", "set_override_model")
builder.add_edge("set_override_model", END)

app = builder.compile()
app.invoke({"config": {}})

{'config': {'model': 'gpt-4o', 'temperature': 0.5}}

In [20]:
def keep_max(cur, new):
    return max(cur or 0, new)

In [25]:
class MultiState(TypedDict):
    messages : Annotated[list, add_messages]
    score : Annotated[int, keep_max]
    meta : Annotated[dict, merge_dicts]
    raw : str

def n1(state):
    return {'message' : [HumanMessage(content='안녕')], 'score' : 50, 'meta' : {'step' : 1}, 'raw' : 'first'}

def n2(state):
    return {'message' : [AIMessage(content='반가워')], 'score' : 80, 'meta' : {'step' : 2, 'ok' : True}, 'raw' : 'second'}

b = StateGraph(MultiState)
b.add_node('n1', n1)
b.add_node('n2', n2)
b.add_edge(START, 'n1')
b.add_edge('n1', 'n2')
b.add_edge('n2', END)

app = b.compile()
app.invoke({"message": [], 'score' : 0, 'meta':{}, 'raw':''})

{'messages': [], 'score': 80, 'meta': {'step': 2, 'ok': True}, 'raw': 'second'}

In [24]:
class TrimState(TypedDict):
    messages : Annotated[list, add_messages]
    visible : list

N = 3

def add_many(state):
    new_msgs = [HumanMessage(content=f"msg-{i}") for i in range(10)]
    return {'messages': new_msgs}

def trim_node(state):
    return {'visible' : state['messages'][-N:]}

b = StateGraph(TrimState)
b.add_node('add_many', add_many)
b.add_node('trim_node', trim_node)
b.add_edge(START, 'add_many')
b.add_edge('add_many', 'trim_node')
b.add_edge('trim_node', END)

app = b.compile()
app.invoke({"message": [], 'visible' : []})

{'messages': [HumanMessage(content='msg-0', additional_kwargs={}, response_metadata={}, id='86f23c78-dc06-4047-be76-cc9df66f60a6'),
  HumanMessage(content='msg-1', additional_kwargs={}, response_metadata={}, id='75a0f76d-d2e6-42b2-87ae-ddebbe158451'),
  HumanMessage(content='msg-2', additional_kwargs={}, response_metadata={}, id='9d093bd5-abf2-46e2-8b82-dcbe828d12d6'),
  HumanMessage(content='msg-3', additional_kwargs={}, response_metadata={}, id='0252db44-5907-4b70-8289-f4f80d8ead57'),
  HumanMessage(content='msg-4', additional_kwargs={}, response_metadata={}, id='6ac401f5-a313-42d1-af8c-53e9772a6c11'),
  HumanMessage(content='msg-5', additional_kwargs={}, response_metadata={}, id='1d5a5785-51ee-456c-9d4f-7ab2df93a095'),
  HumanMessage(content='msg-6', additional_kwargs={}, response_metadata={}, id='eb50db52-aa28-4c64-bbfc-dbe74560dd19'),
  HumanMessage(content='msg-7', additional_kwargs={}, response_metadata={}, id='95e34c38-662a-4e4c-a93c-0a1117d2eb17'),
  HumanMessage(content='msg-

In [26]:
from langgraph.graph.message import RemoveMessage

In [27]:
class TrimState(TypedDict):
    messages : Annotated[list, add_messages]

N = 3

def add_many(state):
    new_msgs = [HumanMessage(content=f"msg-{i}") for i in range(10)]
    return {'messages': new_msgs}

def trim_node(state):
    msgs = state['messages']
    to_remove = msgs[:-N]
    return {'messages' : [RemoveMessage(id=m.id) for m in to_remove]}

b = StateGraph(TrimState)
b.add_node('add_many', add_many)
b.add_node('trim_node', trim_node)
b.add_edge(START, 'add_many')
b.add_edge('add_many', 'trim_node')
b.add_edge('trim_node', END)

app = b.compile()
app.invoke({"message": [], 'visible' : []})

{'messages': [HumanMessage(content='msg-7', additional_kwargs={}, response_metadata={}, id='76f02298-cb75-43bd-bc3e-d9ddba5bbecb'),
  HumanMessage(content='msg-8', additional_kwargs={}, response_metadata={}, id='7b2eadde-2bb4-499a-98fe-2c2a4ca81fa1'),
  HumanMessage(content='msg-9', additional_kwargs={}, response_metadata={}, id='4fa8c6fc-61e8-49bb-bd7b-2aec0d5bd9fb')]}

In [28]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

mixed = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="The capital of France is Paris."),
    ToolMessage(
        content="The capital of France is Paris.", tool_call_id="capital_of_france"
    ),
    HumanMessage(content="What is the capital of England?"),
    AIMessage(content="The capital of England is London."),
    ToolMessage(
        content="The capital of England is London.", tool_call_id="capital_of_england"
    ),
]


class FilterState(TypedDict):
    messages: Annotated[list, add_messages]
    reply: str


def seed(state):
    return {"messages": mixed}


def response(state):
    reply = llm.invoke(drop_tool(state["messages"]))
    return {"reply": reply}

builder = StateGraph(FilterState)
builder.add_node("seed", seed)
builder.add_node("response", response)

builder.add_edge(START, "seed")
builder.add_edge("seed", "response")
builder.add_edge("response", END)

app = builder.compile()
app.invoke({"messages": [], "reply": ""})

NameError: name 'drop_tool' is not defined

In [29]:
class TurnState(TypedDict):
    messages : Annotated[list, add_messages]

def chat_turn(state):
    response = llm.invoke(state['messages'])
    return {'messages' : [response]}

b = StateGraph(TurnState)
b.add_node('chat_turn', chat_turn)

b.add_edge(START, 'chat_turn')
b.add_edge('chat_turn', END)

turn_app = b.compile()

history = [SystemMessage(content='당신은 짧게 대답하는 도우미입니다.')]
for user_text in ['안녕!', '내 이름은 전정이이야', '내 이름이 뭐였지?']:
    out = turn_app.invoke({'messages': history + [HumanMessage(content=user_text)]})
    history = out['messages']
    print(f"USER: {user_text}")
    print(f"BOT : {history[-1].content}")

USER: 안녕!
BOT : 안녕! 어떻게 도와드릴까요?
USER: 내 이름은 전정이이야
BOT : 반가워, 전정이! 무엇을 이야기하고 싶으신가요?
USER: 내 이름이 뭐였지?
BOT : 당신의 이름은 전정이입니다!
